# From Lorenz to AI Weather Models: a minimal AI weather model testbed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<YOUR_GITHUB_USERNAME>/<YOUR_REPO_NAME>/blob/main/integrated_notebook_colab_ready.ipynb)

> **For instructors:** after uploading this notebook to GitHub, replace `<YOUR_GITHUB_USERNAME>` and `<YOUR_REPO_NAME>` in the badge link above with the actual repository path.


## Google Colab setup notes

This notebook is designed to run directly in Google Colab without any extra package installation.

It uses only common scientific Python and PyTorch packages:

- `numpy`
- `matplotlib`
- `torch`
- `torch.nn`
- `TensorDataset` and `DataLoader` from `torch.utils.data`

For this Lorenz-system example, CPU execution is sufficient. A GPU runtime is optional and can be enabled in Colab via **Runtime → Change runtime type → Hardware accelerator → GPU**. The notebook automatically selects GPU if available, otherwise it uses CPU.


In [ ]:
# ============================================================
# 00. Colab / local environment check
# ============================================================
# This cell checks whether the required packages are available.
# It should run without additional installation in Google Colab.
# ============================================================

import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# The notebook uses this device later during PyTorch training.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Reproducibility for the neural-network part
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# 01. Why the Lorenz system matters in meteorology

Weather prediction is an initial-value problem.

This means that to predict the future state of the atmosphere, we need to know its present state as accurately as possible. In practice, however, the atmosphere is never observed perfectly. We always have small uncertainties in temperature, pressure, humidity, wind, and other variables.

The Lorenz system is one of the most famous examples showing why these small uncertainties matter.

Although the Lorenz system is much simpler than the real atmosphere, it captures an important idea:

> In a nonlinear dynamical system, tiny differences in the initial condition can grow rapidly with time.

This is often called **sensitive dependence on initial conditions**, or more popularly, the **butterfly effect**.

In meteorology, this idea is fundamental. It explains why weather forecasts can be accurate for a few days but become increasingly uncertain at longer lead times.

The Lorenz system is therefore useful because it is:

- simple enough to understand and simulate in a short workshop,
- nonlinear, like the atmosphere,
- chaotic for commonly used parameter values,
- useful for demonstrating predictability limits,
- a good toy model for testing numerical integration and machine-learning emulators.

In this notebook, we will use the Lorenz system to study a complete workflow:

1. Simulate a physical dynamical system,
2. Generate training data from that physical system,
3. Train a simple AI-based emulator,
4. Compare physics-based and AI-based time integration.

The goal is not to build a perfect AI model.

The goal is to understand what it means for a neural network to learn the time evolution of a dynamical system, and why this becomes difficult when the system is chaotic.

# 01.1 The Lorenz equations

The Lorenz system consists of three coupled ordinary differential equations.

The three variables are time-dependent and are usually written as:

- $x(t)$
- $y(t)$
- $z(t)$

They evolve with time according to:

$$
\begin{aligned}
\frac{dx}{dt} &= \sigma (y - x) \\
\frac{dy}{dt} &= x(\rho - z) - y \\
\frac{dz}{dt} &= xy - \beta z
\end{aligned}
$$

Here, $x$, $y$, and $z$ are the state variables of the system. In the equations above, we write $x$, $y$, and $z$ instead of $x(t)$, $y(t)$, and $z(t)$ to keep the notation compact.

The parameters are:

- $\sigma$: controls how quickly $x$ responds to the difference between $y$ and $x$
- $\rho$: controls the strength of the thermal forcing
- $\beta$: controls damping in the $z$ equation

Here, \(x\), \(y\), and \(z\) are the state variables of the system.

The parameters are:

- \(\sigma\): controls the rate at which \(x\) responds to differences between \(x\) and \(y\),
- \(\rho\): controls the strength of the thermal forcing,
- \(\beta\): controls the damping of \(z\).

In the original physical interpretation, the Lorenz system was derived as a simplified model of thermal convection.

A useful meteorological analogy is a fluid layer heated from below and cooled from above. When the heating is strong enough, convection can develop. Warm fluid rises, cold fluid sinks, and organized circulation patterns can appear.

The Lorenz system does not represent the full atmosphere. It has only three variables, while the real atmosphere has many interacting variables over three-dimensional space.

However, it is still very useful because it shows how a deterministic system can become unpredictable in practice.

Deterministic means:

> If we know the exact initial condition, the future evolution is fully determined.

Chaotic means:

> If two initial conditions are almost the same, their future trajectories may become very different after some time.

This distinction is central to weather prediction.

### Common parameter choice

A commonly used chaotic version of the Lorenz system uses:

\[
\sigma = 10, \qquad \rho = 28, \qquad \beta = \frac{8}{3}
\]

For these parameter values, the solution often evolves around two preferred regions in phase space. The resulting structure is known as the **Lorenz attractor**.

In our workshop, we will use these standard values because they clearly show chaotic behavior.

Later, we will use numerical integration to simulate the system forward in time.

## 02. Define the Lorenz equations

In [ ]:
# ============================================================
# 02. Define the Lorenz equations
# ============================================================
#
# In this cell, we define the right-hand side of the Lorenz system.
#
# The Lorenz system is written as:
#
#   dx/dt = sigma * (y - x)
#   dy/dt = x * (rho - z) - y
#   dz/dt = x * y - beta * z
#
# A numerical solver will later use this function to compute how the
# system changes at each time step.
#
# This function does NOT yet solve the system.
# It only tells us the instantaneous rate of change.
# ============================================================

import numpy as np

def lorenz_rhs(state, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Compute the right-hand side of the Lorenz system.

    Parameters
    ----------
    state: array-like of shape (3,)
        Current state of the system.
        state[0] = x
        state[1] = y
        state[2] = z

    sigma: float
        Lorenz parameter controlling the response of x to y - x.

    rho: float
        Lorenz parameter related to thermal forcing.

    beta: float
        Lorenz parameter controlling damping in the z equation.

    Returns
    -------
    derivatives: numpy.ndarray of shape (3,)
        Time derivatives [dx/dt, dy/dt, dz/dt].
    """

    # Unpack the current state
    x, y, z = state

    # Lorenz equations
    dxdt = sigma * (y - x)
    dydt = x * (rho - z) - y
    dzdt = x * y - beta * z

    # Return the derivatives as a NumPy array
    return np.array([dxdt, dydt, dzdt])

In [ ]:
# Let us test the function at one initial condition.

initial_state = np.array([1.0, 1.0, 1.0])

derivatives = lorenz_rhs(initial_state)

print("Initial state:")
print(initial_state)

print("\nTime derivatives at the initial state:")
print(derivatives)

The output gives the instantaneous tendency of the system at the chosen initial condition.

For example, if `dx/dt` is positive, then `x` is increasing at that moment.
If `dz/dt` is negative, then `z` is decreasing at that moment.

In the next section, we will use these tendencies to step the system forward in time.

# 03. Numerical time integration

The Lorenz equations tell us the instantaneous rate of change of the system:

$$
\frac{d\mathbf{x}}{dt} = f(\mathbf{x})
$$

where the state vector is

$$
\mathbf{x}(t) =
\begin{bmatrix}
x(t) \\
y(t) \\
z(t)
\end{bmatrix}
$$

In continuous time, the system evolves smoothly. But on a computer, we cannot solve the system at every possible instant. Instead, we approximate the evolution using small time steps.

We choose a time step $\Delta t$ and compute the state at discrete times:

$$
t_0, \ t_1, \ t_2, \dots
$$

where

$$
t_{n+1} = t_n + \Delta t
$$

The goal of numerical time integration is to estimate the next state:

$$
\mathbf{x}_{n+1}
$$

from the current state:

$$
\mathbf{x}_n
$$

## Forward Euler method

The simplest time-stepping method is the Forward Euler method.

The idea is:

> current state + small step in the direction of the current tendency

Mathematically:

$$
\mathbf{x}_{n+1}
=
\mathbf{x}_n
+
\Delta t \ f(\mathbf{x}_n)
$$

For the Lorenz system, this means:

$$
x_{n+1} = x_n + \Delta t \frac{dx}{dt}
$$

$$
y_{n+1} = y_n + \Delta t \frac{dy}{dt}
$$

$$
z_{n+1} = z_n + \Delta t \frac{dz}{dt}
$$

Euler's method is easy to understand and useful for teaching. However, it is not always very accurate, especially for chaotic systems.

Later, we will also use a more accurate method called fourth-order Runge-Kutta, or RK4.

## Euler time step

In [ ]:
# ============================================================
# 03. Numerical time integration: Forward Euler method
# ============================================================
#
# We already defined lorenz_rhs(state), which computes the
# instantaneous derivatives:
#
#   dx/dt, dy/dt, dz/dt
#
# Now we define a function that takes one numerical time step.
#
# Forward Euler formula:
#
#   next_state = current_state + dt * derivative
#
# where:
#
#   current_state = [x, y, z] at the current time
#   derivative    = [dx/dt, dy/dt, dz/dt]
#   dt            = time step size
# ============================================================

def euler_step(state, dt, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Take one Forward Euler time step for the Lorenz system.

    Parameters
    ----------
    state : numpy.ndarray of shape (3,)
        Current state [x, y, z].

    dt : float
        Time step size.

    sigma, rho, beta : float
        Lorenz system parameters.

    Returns
    -------
    next_state : numpy.ndarray of shape (3,)
        Estimated state after one time step.
    """

    # Compute the derivatives at the current state
    derivatives = lorenz_rhs(state, sigma=sigma, rho=rho, beta=beta)

    # Move the system forward by one small time step
    next_state = state + dt * derivatives

    return next_state

## test one Euler step

In [ ]:
# Let us test one Euler step.

state_now = np.array([1.0, 1.0, 1.0])
dt = 0.01

state_next = euler_step(state_now, dt)

print("Current state:")
print(state_now)

print("\nState after one Euler step:")
print(state_next)

The result is the estimated state after one small time step.

This is only one step. To simulate the Lorenz system over a longer time, we need to repeat this operation many times.

For example, if we want to simulate from $t=0$ to $t=20$ using $\Delta t = 0.01$, then the number of time steps is:

$$
N = \frac{20}{0.01} = 2000
$$

So the computer repeatedly applies:

$$
\mathbf{x}_{n+1}
=
\mathbf{x}_n
+
\Delta t \ f(\mathbf{x}_n)
$$

## simulate a full trajectory using Euler

In [ ]:
# ============================================================
# Simulate a full Lorenz trajectory using Euler integration
# ============================================================
#
# This function repeatedly applies the Euler step.
#
# Starting from an initial condition, it creates a full trajectory:
#
#   state at time 0
#   state at time 1
#   state at time 2
#   ...
#
# In array form, the output will have shape:
#
#   (number_of_steps + 1, 3)
#
# The +1 is because we also store the initial condition.
# ============================================================

def simulate_lorenz_euler(initial_state, dt=0.01, total_time=20.0,
                          sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Simulate the Lorenz system using Forward Euler integration.

    Parameters
    ----------
    initial_state : numpy.ndarray of shape (3,)
        Initial condition [x0, y0, z0].

    dt : float
        Time step size.

    total_time : float
        Total simulation time.

    sigma, rho, beta : float
        Lorenz system parameters.

    Returns
    -------
    times : numpy.ndarray of shape (num_steps + 1,)
        Time values.

    trajectory : numpy.ndarray of shape (num_steps + 1, 3)
        Simulated Lorenz trajectory.
        Column 0 contains x(t).
        Column 1 contains y(t).
        Column 2 contains z(t).
    """

    # Compute the number of numerical time steps
    num_steps = int(total_time / dt)

    # Create an array of time values
    times = np.linspace(0.0, total_time, num_steps + 1)

    # Create an empty array to store the trajectory
    trajectory = np.zeros((num_steps + 1, 3))

    # Store the initial condition
    trajectory[0] = initial_state

    # Repeatedly apply the Euler method
    for n in range(num_steps):
        trajectory[n + 1] = euler_step(
            trajectory[n],
            dt,
            sigma=sigma,
            rho=rho,
            beta=beta
        )

    return times, trajectory

## run the simulation

In [ ]:
# Initial condition
initial_state = np.array([1.0, 1.0, 1.0])

# Time-step settings
dt = 0.01
total_time = 40.0

# Run the simulation
times, trajectory = simulate_lorenz_euler(
    initial_state=initial_state,
    dt=dt,
    total_time=total_time
)

print("Shape of times array:")
print(times.shape)

print("\nShape of trajectory array:")
print(trajectory.shape)

print("\nFirst five states:")
print(trajectory[:5])

The trajectory array has three columns:

- column 0: $x(t)$
- column 1: $y(t)$
- column 2: $z(t)$

Each row corresponds to one time step.

Now that we have simulated the Lorenz system, the next step is to visualize the solution.

## 04. Visualize the Lorenz trajectory

In [ ]:
# ============================================================
# Plot the Lorenz variables as time series
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.plot(times, trajectory[:, 0], label="x(t)")
plt.plot(times, trajectory[:, 1], label="y(t)")
plt.plot(times, trajectory[:, 2], label="z(t)")

plt.xlabel("Time")
plt.ylabel("State value")
plt.title("Lorenz system time series using Euler integration")
plt.legend()
plt.grid(True)
plt.show()

## plot 3D trajectory

In [ ]:
# ============================================================
# Plot the Lorenz attractor in 3D phase space
# ============================================================
#
# The Lorenz system has three variables: x, y, and z.
# Instead of plotting each variable against time, we can plot
# the trajectory in three-dimensional phase space.
#
# Each point in this plot is one system state:
#
#   [x(t), y(t), z(t)]
#
# The curve shows how the system moves through state space.
# ============================================================

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    trajectory[:, 0],
    trajectory[:, 1],
    trajectory[:, 2],
    linewidth=0.7
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Lorenz attractor using Euler integration")

plt.show()

# 05. Generate training data from the physics model

We now have a numerical model of the Lorenz system.

This numerical model acts as our "physics model". It can generate trajectories by integrating the Lorenz equations forward in time.

For machine learning, we need training examples.

A simple way to create training data is to use pairs of consecutive states:

$$
\mathbf{x}_n \rightarrow \mathbf{x}_{n+1}
$$

The input to the neural network will be the current state:

$$
\mathbf{x}_n = [x_n, y_n, z_n]
$$

The target output will be the next state:

$$
\mathbf{x}_{n+1} = [x_{n+1}, y_{n+1}, z_{n+1}]
$$

So the neural network learns a discrete time-step map:

$$
\mathcal{M}_{AI}(\mathbf{x}_n) \approx \mathbf{x}_{n+1}
$$

This is similar to what a numerical time-integration scheme does.

However, instead of explicitly using the Lorenz equations, the neural network learns the mapping from data generated by the physics model.

## Why do we need many initial conditions?

If we train the neural network using only one trajectory, it may only learn the behavior along that specific path.

To make the model more general, we generate many short Lorenz trajectories from different initial conditions.

Each trajectory gives us many input-target pairs.

For example, from one trajectory:

$$
\mathbf{x}_0, \mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_N
$$

we can create the pairs:

$$
(\mathbf{x}_0, \mathbf{x}_1)
$$

$$
(\mathbf{x}_1, \mathbf{x}_2)
$$

$$
(\mathbf{x}_2, \mathbf{x}_3)
$$

and so on.

The neural network then sees many examples of how the system evolves in different parts of phase space.

## 05.1 A more accurate time-stepper: RK4

The Forward Euler method uses only the tendency at the beginning of the time step.

RK4, or fourth-order Runge-Kutta, is more accurate because it estimates the tendency several times within one time step.

Conceptually, RK4 asks:

1. What is the tendency at the start of the step?
2. What is the tendency halfway through the step?
3. What is another estimate halfway through the step?
4. What is the tendency at the end of the step?

Then it combines these estimates to produce a better next state.

For chaotic systems such as the Lorenz system, numerical accuracy matters. Therefore, we will use RK4 to generate the training data.

## RK4 step

In [ ]:
# ============================================================
# 05.1 More accurate numerical integration: RK4
# ============================================================
#
# RK4 stands for fourth-order Runge-Kutta.
#
# It is a standard numerical method for solving ordinary
# differential equations.
#
# Compared with Forward Euler, RK4 is usually much more accurate
# for the same time step size.
#
# We will use RK4 as the "physics model" for generating training data.
# ============================================================

def rk4_step(state, dt, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Take one fourth-order Runge-Kutta time step for the Lorenz system.

    Parameters
    ----------
    state: numpy.ndarray of shape (3,)
        Current state [x, y, z].

    dt: float
        Time step size.

    sigma, rho, beta: float
        Lorenz system parameters.

    Returns
    -------
    next_state: numpy.ndarray of shape (3,)
        Estimated state after one RK4 time step.
    """

    # First tendency estimate: at the beginning of the interval
    k1 = lorenz_rhs(state, sigma=sigma, rho=rho, beta=beta)

    # Second tendency estimate: halfway through the interval,
    # using k1 to estimate the midpoint state
    k2 = lorenz_rhs(state + 0.5 * dt * k1, sigma=sigma, rho=rho, beta=beta)

    # Third tendency estimate: another midpoint estimate,
    # now using k2
    k3 = lorenz_rhs(state + 0.5 * dt * k2, sigma=sigma, rho=rho, beta=beta)

    # Fourth tendency estimate: at the end of the interval,
    # using k3 to estimate the endpoint state
    k4 = lorenz_rhs(state + dt * k3, sigma=sigma, rho=rho, beta=beta)

    # Weighted average of the four tendency estimates
    next_state = state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

    return next_state

## 05.2 Simulate a trajectory using RK4

In [ ]:
# ============================================================
# Simulate a full Lorenz trajectory using RK4 integration
# ============================================================

def simulate_lorenz_rk4(initial_state, dt=0.01, total_time=20.0,
                        sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Simulate the Lorenz system using RK4 integration.

    Parameters
    ----------
    initial_state: numpy.ndarray of shape (3,)
        Initial condition [x0, y0, z0].

    dt: float
        Time step size.

    total_time : float
        Total simulation time.

    sigma, rho, beta: float
        Lorenz system parameters.

    Returns
    -------
    times: numpy.ndarray of shape (num_steps + 1,)
        Time values.

    trajectory: numpy.ndarray of shape (num_steps + 1, 3)
        Simulated Lorenz trajectory.
    """

    # Number of time steps
    num_steps = int(total_time / dt)

    # Time values
    times = np.linspace(0.0, total_time, num_steps + 1)

    # Array for storing the trajectory
    trajectory = np.zeros((num_steps + 1, 3))

    # Store initial condition
    trajectory[0] = initial_state

    # Time integration loop
    for n in range(num_steps):
        trajectory[n + 1] = rk4_step(
            trajectory[n],
            dt,
            sigma=sigma,
            rho=rho,
            beta=beta
        )

    return times, trajectory

In [ ]:
# Let us compare Euler and RK4 for the same initial condition.

initial_state = np.array([1.0, 1.0, 1.0])
dt = 0.01
total_time = 20.0

times_euler, trajectory_euler = simulate_lorenz_euler(
    initial_state=initial_state,
    dt=dt,
    total_time=total_time
)

times_rk4, trajectory_rk4 = simulate_lorenz_rk4(
    initial_state=initial_state,
    dt=dt,
    total_time=total_time
)

plt.figure(figsize=(12, 5))

plt.plot(times_euler, trajectory_euler[:, 0], label="Euler: x(t)", alpha=0.8)
plt.plot(times_rk4, trajectory_rk4[:, 0], label="RK4: x(t)", alpha=0.8)

plt.xlabel("Time")
plt.ylabel("x(t)")
plt.title("Euler and RK4 can gradually diverge for a chaotic system")
plt.legend()
plt.grid(True)
plt.show()

The Euler and RK4 trajectories may look similar at first, but they can separate later.

This does not necessarily mean one solution is physically "wrong" in a simple sense. In chaotic systems, even very small numerical differences can grow rapidly.

However, RK4 is generally a more accurate time-integration method than Forward Euler for the same time step.

For the rest of the notebook, we will use RK4-generated trajectories as our reference physics data.

# 05.3 Sensitivity to initial conditions

The Lorenz system is an initial-value problem.

This means that the future evolution of the system depends on the initial condition:

$$
\mathbf{x}(0) =
\begin{bmatrix}
x(0) \\
y(0) \\
z(0)
\end{bmatrix}
$$

If we know the initial condition exactly, then the Lorenz equations determine the future trajectory.

However, in chaotic systems, very small differences in the initial condition can grow rapidly with time.

This is called **sensitive dependence on initial conditions**.

In meteorology, this is extremely important. The atmosphere is also an initial-value problem. Weather forecasts start from an estimate of the current atmospheric state, but this estimate is never perfect. Small uncertainties in the initial atmospheric state can grow and eventually lead to large forecast differences.

This is one of the reasons why weather forecasts become less certain at longer lead times.

## Two almost identical initial conditions

We now simulate two Lorenz trajectories.

The first trajectory starts from:

$$
\mathbf{x}_0 =
\begin{bmatrix}
1.0 \\
1.0 \\
1.0
\end{bmatrix}
$$

The second trajectory starts from an almost identical initial condition:

$$
\mathbf{x}_0^\prime =
\begin{bmatrix}
1.0 + 10^{-6} \\
1.0 \\
1.0
\end{bmatrix}
$$

The difference between the two initial states is extremely small.

At first, the two trajectories should remain very close.

Later, the difference can grow rapidly.

In [ ]:
# ============================================================
# Sensitivity to initial conditions
# ============================================================
#
# We now simulate two Lorenz trajectories that start from almost
# the same initial condition.
#
# This demonstrates sensitive dependence on initial conditions:
#
#   tiny initial difference -> large difference later
#
# This is one of the most important lessons of the Lorenz system.
# ============================================================

# Two nearly identical initial conditions
initial_state_1 = np.array([1.0, 1.0, 1.0])
initial_state_2 = np.array([1.0 + 1e-6, 1.0, 1.0])

# Simulation settings
dt = 0.01
total_time = 30.0

# Simulate both trajectories using RK4
times, trajectory_1 = simulate_lorenz_rk4(
    initial_state=initial_state_1,
    dt=dt,
    total_time=total_time
)

_, trajectory_2 = simulate_lorenz_rk4(
    initial_state=initial_state_2,
    dt=dt,
    total_time=total_time
)

In [ ]:
# ============================================================
# Plot x(t) for the two nearly identical initial conditions
# ============================================================

plt.figure(figsize=(12, 5))

plt.plot(times, trajectory_1[:, 0], label="Trajectory 1: x(t)")
plt.plot(times, trajectory_2[:, 0], "--", label="Trajectory 2: x(t)")

plt.xlabel("Time")
plt.ylabel("x(t)")
plt.title("Two Lorenz trajectories with nearly identical initial conditions")
plt.legend()
plt.grid(True)
plt.show()

At early times, the two curves may lie almost on top of each other.

After some time, they begin to separate.

This does not happen because the system is random. The Lorenz system is fully deterministic.

The separation happens because the dynamics are nonlinear and chaotic.

In [ ]:
# ============================================================
# Compute the distance between the two trajectories
# ============================================================
#
# At each time step, we compute the Euclidean distance:
#
#   distance = sqrt((x1 - x2)^2 + (y1 - y2)^2 + (z1 - z2)^2)
#
# This gives one number measuring how far apart the two states are.
# ============================================================

difference = trajectory_1 - trajectory_2

distance = np.linalg.norm(difference, axis=1)

plt.figure(figsize=(10, 5))

plt.plot(times, distance, linewidth=2)

plt.xlabel("Time")
plt.ylabel("Distance between trajectories")
plt.title("Growth of initial-condition error")
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# Plot the error growth on a logarithmic scale
# ============================================================
#
# A logarithmic y-axis helps us see the early exponential-like
# growth of small errors.
# ============================================================

plt.figure(figsize=(10, 5))

plt.semilogy(times, distance, linewidth=2)

plt.xlabel("Time")
plt.ylabel("Distance between trajectories, log scale")
plt.title("Error growth from a tiny initial perturbation")
plt.grid(True)
plt.show()

The logarithmic plot often shows an approximately straight-line region at early times.

This indicates exponential-like error growth.

This is one of the signatures of chaos.

In weather prediction, a similar idea applies:

- we never know the atmospheric initial condition perfectly,
- small initial errors can grow,
- after some lead time, the exact forecast trajectory becomes unreliable.

This is why ensemble prediction is important in meteorology.

Instead of making only one forecast, we run several forecasts from slightly different initial conditions. The spread between the forecasts gives information about uncertainty.

In [ ]:
# ============================================================
# Plot both trajectories in 3D phase space
# ============================================================

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    trajectory_1[:, 0],
    trajectory_1[:, 1],
    trajectory_1[:, 2],
    label="Trajectory 1",
    linewidth=1.2
)

ax.plot(
    trajectory_2[:, 0],
    trajectory_2[:, 1],
    trajectory_2[:, 2],
    "--",
    label="Trajectory 2",
    linewidth=1.2
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Sensitive dependence on initial conditions")
ax.legend()

plt.show()

## 05.4 Creating input-target pairs

Now we create the dataset for machine learning.

For every trajectory, we take consecutive states.

Input:

$$
\mathbf{x}_n
$$

Target:

$$
\mathbf{x}_{n+1}
$$

The neural network will learn:

$$
\mathbf{x}_{n+1} \approx \mathcal{M}_{AI}(\mathbf{x}_n)
$$

This is a one-step prediction problem.

Later, to make a full AI-based trajectory, we will repeatedly apply the neural network:

$$
\mathbf{x}_0
\rightarrow
\mathbf{x}_1
\rightarrow
\mathbf{x}_2
\rightarrow
\dots
$$

This is called an iterative rollout.

## Generate many Lorenz trajectories from random initial states

In [ ]:
# ============================================================
# Generate many Lorenz trajectories from random initial states
# ============================================================
#
# This function creates a machine-learning dataset.
#
# Each training example is:
#
#   input  = current state  [x_n, y_n, z_n]
#   target = next state     [x_{n+1}, y_{n+1}, z_{n+1}]
#
# We generate many short trajectories from different initial states.
# This helps the neural network see different regions of phase space.
# ============================================================

def generate_lorenz_dataset(num_trajectories=200,
                            steps_per_trajectory=300,
                            dt=0.01,
                            initial_range=(-20.0, 20.0),
                            sigma=10.0,
                            rho=28.0,
                            beta=8.0/3.0,
                            seed=42):
    """
    Generate supervised learning data from the Lorenz system.

    Parameters
    ----------
    num_trajectories : int
        Number of independent trajectories to simulate.

    steps_per_trajectory : int
        Number of time steps in each trajectory.

    dt: float
        Time step size.

    initial_range: tuple of float
        Range from which initial x, y, z values are sampled uniformly.

    sigma, rho, beta: float
        Lorenz parameters.

    seed: int
        Random seed for reproducibility.

    Returns
    -------
    X: numpy.ndarray of shape (num_samples, 3)
        Input states.

    Y: numpy.ndarray of shape (num_samples, 3)
        Target next states.
    """

    # Set random seed so that results are reproducible
    rng = np.random.default_rng(seed)

    # Lists for storing input and target states
    inputs = []
    targets = []

    # Loop over independent trajectories
    for i in range(num_trajectories):

        # Random initial condition
        initial_state = rng.uniform(
            low=initial_range[0],
            high=initial_range[1],
            size=3
        )

        # We simulate manually here because we want exactly
        # steps_per_trajectory input-target pairs.
        state = initial_state.copy()

        for n in range(steps_per_trajectory):

            # Compute next state using the physics model
            next_state = rk4_step(
                state,
                dt,
                sigma=sigma,
                rho=rho,
                beta=beta
            )

            # Store one supervised learning pair
            inputs.append(state)
            targets.append(next_state)

            # Move forward in time
            state = next_state

    # Convert lists to NumPy arrays
    X = np.array(inputs, dtype=np.float32)
    Y = np.array(targets, dtype=np.float32)

    return X, Y

## Create the dataset

In [ ]:
# Generate the dataset

X, Y = generate_lorenz_dataset(
    num_trajectories=200,
    steps_per_trajectory=300,
    dt=0.01,
    initial_range=(-20.0, 20.0),
    seed=42
)

print("Input data shape:", X.shape)
print("Target data shape:", Y.shape)

print("\nFirst input state:")
print(X[0])

print("\nFirst target next state:")
print(Y[0])

## 05.5 Visualizing the training data

Before training a neural network, it is useful to inspect the data.

Here we plot the input states in phase space.

Each point is one state sampled from the physics-generated trajectories.

## 3D scatter plot of training states

In [ ]:
# ============================================================
# Visualize the training data in phase space
# ============================================================

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

# Plot only a subset of points to keep the figure readable
num_points_to_plot = 5000

ax.scatter(
    X[:num_points_to_plot, 0],
    X[:num_points_to_plot, 1],
    X[:num_points_to_plot, 2],
    s=2,
    alpha=0.4
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Sample of physics-generated training states")

plt.show()

## 05.6 Train/validation split

We split the dataset into two parts:

1. **Training data**  
   Used to update the neural network weights.

2. **Validation data**  
   Used to check whether the model performs well on data not directly used for training.

This helps us detect whether the neural network is learning a useful mapping, rather than only memorizing the training examples.

In [ ]:
# ============================================================
# Split the dataset into training and validation sets
# ============================================================

def train_validation_split(X, Y, validation_fraction=0.2, seed=42):
    """
    Split input-target pairs into training and validation sets.

    Parameters
    ----------
    X: numpy.ndarray
        Input states.

    Y: numpy.ndarray
        Target states.

    validation_fraction : float
        Fraction of samples used for validation.

    seed: int
        Random seed for reproducibility.

    Returns
    -------
    X_train, Y_train, X_val, Y_val : numpy.ndarray
        Training and validation arrays.
    """

    rng = np.random.default_rng(seed)

    num_samples = X.shape[0]

    # Create a shuffled list of indices
    indices = np.arange(num_samples)
    rng.shuffle(indices)

    # Number of validation samples
    num_val = int(validation_fraction * num_samples)

    # Validation indices
    val_indices = indices[:num_val]

    # Training indices
    train_indices = indices[num_val:]

    # Create splits
    X_train = X[train_indices]
    Y_train = Y[train_indices]

    X_val = X[val_indices]
    Y_val = Y[val_indices]

    return X_train, Y_train, X_val, Y_val


X_train, Y_train, X_val, Y_val = train_validation_split(
    X,
    Y,
    validation_fraction=0.2,
    seed=42
)

print("Training input shape:", X_train.shape)
print("Training target shape:", Y_train.shape)

print("\nValidation input shape:", X_val.shape)
print("Validation target shape:", Y_val.shape)

At this point, we have a complete supervised learning dataset.

The physics model generated examples of the form:

$$
\mathbf{x}_n \rightarrow \mathbf{x}_{n+1}
$$

The next step is to build a simple neural network emulator in PyTorch.

The emulator will learn the one-step Lorenz time integration map.

# 06. Build a simple AI-based emulator

We now build a simple neural network emulator for the Lorenz system.

The emulator receives the current state of the system:

$$
\mathbf{x}_n = [x_n, y_n, z_n]
$$

and predicts the next state:

$$
\mathbf{x}_{n+1} = [x_{n+1}, y_{n+1}, z_{n+1}]
$$

So the neural network learns the mapping:

$$
\mathcal{M}_{AI}(\mathbf{x}_n) \approx \mathbf{x}_{n+1}
$$

This is a supervised learning problem.

Input size: 3  
Output size: 3

The model does not know the Lorenz equations explicitly. It only sees examples generated by the physics model.

## Why use a small neural network?

The Lorenz system has only three variables, so we do not need a large model.

A small fully connected neural network is enough for this workshop.

The model will have:

- an input layer with 3 values: $x$, $y$, and $z$
- a few hidden layers with nonlinear activation functions
- an output layer with 3 values: predicted next $x$, $y$, and $z$

The nonlinear activation function is important because the Lorenz system itself is nonlinear.

In [ ]:
# ============================================================
# 06. Build a simple AI-based emulator
# ============================================================
#
# We will use PyTorch to build and train the neural network.
#
# PyTorch is commonly used for deep learning research.
# In Google Colab, we can run this on CPU or GPU.
#
# For this small Lorenz example, CPU is already enough.
# But if a GPU is available, PyTorch can use it automatically.
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Choose the computing device
# "cuda" means GPU
# "cpu" means central processor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

## 06.1 Data normalization

Before training a neural network, it is usually helpful to normalize the data.

The Lorenz variables can have different ranges. Neural networks often train more smoothly if the input and target values are scaled.

We will compute the mean and standard deviation from the training input data:

$$
\mu = \text{mean}(\mathbf{x})
$$

$$
\sigma_x = \text{standard deviation}(\mathbf{x})
$$

Then we normalize both inputs and targets as:

$$
\mathbf{x}_{norm} = \frac{\mathbf{x} - \mu}{\sigma_x}
$$

For this simple one-step prediction task, we use the same normalization statistics for both input states and target states.

This is reasonable because both inputs and targets are Lorenz states.

In [ ]:
# ============================================================
# Normalize the Lorenz dataset
# ============================================================
#
# We compute the mean and standard deviation from the training inputs.
#
# Important:
# We should not compute normalization statistics using the validation set.
# The validation set should represent unseen data.
# ============================================================

# Compute mean and standard deviation using training inputs only
state_mean = X_train.mean(axis=0)
state_std = X_train.std(axis=0)

# Add a tiny value to avoid division by zero
state_std = state_std + 1e-8

print("State mean:")
print(state_mean)

print("\nState standard deviation:")
print(state_std)


def normalize_state(x):
    """
    Normalize Lorenz states using training-set statistics.
    """
    return (x - state_mean) / state_std


def denormalize_state(x_norm):
    """
    Convert normalized Lorenz states back to physical units.
    """
    return x_norm * state_std + state_mean


# Normalize the data
X_train_norm = normalize_state(X_train)
Y_train_norm = normalize_state(Y_train)

X_val_norm = normalize_state(X_val)
Y_val_norm = normalize_state(Y_val)

print("\nExample before normalization:")
print(X_train[0])

print("\nExample after normalization:")
print(X_train_norm[0])

## 06.2 PyTorch datasets and dataloaders

PyTorch expects data in tensor form.

A tensor is similar to a NumPy array, but it can be used efficiently by PyTorch and can be moved to a GPU.

We will create:

- a training dataset,
- a validation dataset,
- a training dataloader,
- a validation dataloader.

The dataloader gives the neural network small batches of data during training.

Instead of processing all examples at once, the model sees mini-batches, for example 256 samples at a time.

In [ ]:
# ============================================================
# Convert NumPy arrays to PyTorch tensors
# ============================================================

# Convert normalized arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
Y_train_tensor = torch.tensor(Y_train_norm, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_norm, dtype=torch.float32)
Y_val_tensor = torch.tensor(Y_val_norm, dtype=torch.float32)

# Create PyTorch datasets
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

# Batch size controls how many samples are processed at once
batch_size = 256

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))

## 06.3 Neural network architecture

We use a small multilayer perceptron, also called an MLP.

An MLP is a neural network made of fully connected layers.

Here the architecture is:

$$
3 \rightarrow 32 \rightarrow 32 \rightarrow 32 \rightarrow 3
$$

This means:

- input: 3 Lorenz variables
- hidden layer 1: 32 neurons
- hidden layer 2: 32 neurons
- hidden layer 3: 32 neurons
- output: 3 predicted Lorenz variables

We use the `Tanh` activation function because it is smooth and works well for this small continuous dynamical system.

In [ ]:
# ============================================================
# Define a simple neural network emulator
# ============================================================
#
# The model receives:
#
#   [x_n, y_n, z_n]
#
# and predicts:
#
#   [x_{n+1}, y_{n+1}, z_{n+1}]
#
# Since the Lorenz system is nonlinear, we use nonlinear
# activation functions between the linear layers.
# ============================================================

class LorenzEmulator(nn.Module):
    """
    Simple neural network emulator for one-step Lorenz prediction.
    """

    def __init__(self, hidden_dim=32):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),

            nn.Linear(hidden_dim, 3)
        )

    def forward(self, x):
        """
        Forward pass through the neural network.

        Parameters
        ----------
        x : torch.Tensor of shape (batch_size, 3)
            Normalized current Lorenz states.

        Returns
        -------
        torch.Tensor of shape (batch_size, 3)
            Normalized predicted next Lorenz states.
        """
        return self.network(x)


# Create the model and move it to CPU or GPU
model = LorenzEmulator(hidden_dim=64).to(device)

print(model)

![Lorenz MLP architecture](nn.svg)

## 06.4 Loss function and optimizer

The neural network produces a prediction:

$$
\hat{\mathbf{x}}_{n+1}
$$

The true target from the physics model is:

$$
\mathbf{x}_{n+1}
$$

We need to measure how different they are.

We use the mean squared error loss:

$$
\text{MSE}
=
\frac{1}{N}
\sum_i
\left(
\hat{\mathbf{x}}_i - \mathbf{x}_i
\right)^2
$$

The optimizer then adjusts the neural network weights to reduce this loss.

Here we use the Adam optimizer, which is a commonly used optimizer for neural networks.

In [ ]:
# ============================================================
# Define loss function and optimizer
# ============================================================

# Mean squared error loss
loss_fn = nn.MSELoss()

# Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# 07. Train the AI emulator

Training means repeatedly showing the neural network examples of the form:

$$
\mathbf{x}_n \rightarrow \mathbf{x}_{n+1}
$$

For each mini-batch, PyTorch performs the following steps:

1. send input states through the neural network,
2. compute the prediction error,
3. compute gradients using backpropagation,
4. update the model weights using the optimizer.

We repeat this process for several epochs.

One epoch means that the model has seen the full training dataset once.

In [ ]:
# ============================================================
# Training loop
# ============================================================
#
# This cell trains the neural network-emulator.
#
# During training:
#
# 1. The model predicts the next state.
# 2. We compare the prediction with the true next state.
# 3. The loss measures the error.
# 4. Backpropagation computes gradients.
# 5. The optimizer updates the model weights.
# ============================================================

num_epochs = 30

train_losses = []
val_losses = []

for epoch in range(num_epochs):

    # -------------------------
    # Training phase
    # -------------------------
    model.train()

    running_train_loss = 0.0

    for batch_X, batch_Y in train_loader:

        # Move batch data to CPU or GPU
        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)

        # Reset gradients from previous batch
        optimizer.zero_grad()

        # Forward pass: predict next state
        predictions = model(batch_X)

        # Compute loss
        loss = loss_fn(predictions, batch_Y)

        # Backward pass: compute gradients
        loss.backward()

        # Update model weights
        optimizer.step()

        # Store loss contribution
        running_train_loss += loss.item() * batch_X.size(0)

    # Average training loss over all training samples
    epoch_train_loss = running_train_loss / len(train_dataset)

    # -------------------------
    # Validation phase
    # -------------------------
    model.eval()

    running_val_loss = 0.0

    # We do not need gradients during validation
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:

            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)

            predictions = model(batch_X)

            loss = loss_fn(predictions, batch_Y)

            running_val_loss += loss.item() * batch_X.size(0)

    # Average validation loss over all validation samples
    epoch_val_loss = running_val_loss / len(val_dataset)

    # Store losses
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)

    # Print progress
    print(
        f"Epoch {epoch+1:03d}/{num_epochs} | "
        f"Train loss: {epoch_train_loss:.6e} | "
        f"Validation loss: {epoch_val_loss:.6e}"
    )

## 07.1 Training curve

The training curve shows how the loss changes during training.

A good sign is that both training loss and validation loss decrease.

If the training loss decreases but the validation loss increases, the model may be overfitting.

For this simple Lorenz example, both losses should usually decrease smoothly.

In [ ]:
# ============================================================
# Plot training and validation losses
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Training loss")
plt.plot(val_losses, label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Mean squared error")
plt.title("Training curve for the Lorenz emulator")
plt.yscale("log")
plt.legend()
plt.grid(True)
plt.show()

## 07.2 One-step prediction test

The model was trained to predict one time step ahead.

Before doing a long rollout, we first test a simple one-step prediction.

We take one validation input:

$$
\mathbf{x}_n
$$

The true target is:

$$
\mathbf{x}_{n+1}
$$

The model predicts:

$$
\hat{\mathbf{x}}_{n+1}
$$

We compare the true next state and the predicted next state in physical units.

In [ ]:
# ============================================================
# Test one-step prediction on a validation example
# ============================================================

# Pick one example from the validation set
example_index = 0

x_input = X_val[example_index]
y_true = Y_val[example_index]

# Normalize input
x_input_norm = normalize_state(x_input)

# Convert to PyTorch tensor
x_input_tensor = torch.tensor(
    x_input_norm,
    dtype=torch.float32
).unsqueeze(0).to(device)

# Predict with the trained model
model.eval()
with torch.no_grad():
    y_pred_norm_tensor = model(x_input_tensor)

# Convert prediction back to NumPy
y_pred_norm = y_pred_norm_tensor.cpu().numpy()[0]

# Denormalize prediction back to physical Lorenz units
y_pred = denormalize_state(y_pred_norm)

print("Input state x_n:")
print(x_input)

print("\nTrue next state x_{n+1}:")
print(y_true)

print("\nPredicted next state:")
print(y_pred)

print("\nPrediction error:")
print(y_pred - y_true)

The one-step prediction should usually be quite accurate.

However, this does not yet mean that the neural network can produce a perfect long trajectory.

For a long AI-based simulation, we will feed the model's own prediction back into itself:

$$
\mathbf{x}_0
\rightarrow
\hat{\mathbf{x}}_1
\rightarrow
\hat{\mathbf{x}}_2
\rightarrow
\hat{\mathbf{x}}_3
\rightarrow
\dots
$$

Small one-step errors can accumulate.

In a chaotic system, these errors can grow rapidly.

This is what we will examine next.

# 08. AI-based rollout and comparison with physics

So far, the neural network emulator has been trained for **one-step prediction**.

It receives the current state:

$$
\mathbf{x}_n
$$

and predicts the next state:

$$
\hat{\mathbf{x}}_{n+1}
$$

Now we want to use the neural network as a time-integration model.

This means we start from an initial condition and repeatedly apply the emulator:

$$
\mathbf{x}_0
\rightarrow
\hat{\mathbf{x}}_1
\rightarrow
\hat{\mathbf{x}}_2
\rightarrow
\hat{\mathbf{x}}_3
\rightarrow
\dots
$$

This is called a **rollout**.

The important difference is:

- during one-step prediction, the model receives true physics-generated inputs,
- during rollout, the model receives its own previous predictions as input.

This makes rollout much harder.

If the model makes a small error at one step, that error becomes part of the next input. In a chaotic system like the Lorenz system, these small errors can grow rapidly.

## Physics rollout versus AI rollout

We will now compare two trajectories starting from the same initial condition.

### Physics rollout

The physics rollout uses the RK4 numerical solver applied to the Lorenz equations:

$$
\mathbf{x}_{n+1} = \mathcal{M}_{physics}(\mathbf{x}_n)
$$

### AI rollout

The AI rollout uses the trained neural network emulator:

$$
\hat{\mathbf{x}}_{n+1} = \mathcal{M}_{AI}(\hat{\mathbf{x}}_n)
$$

Both start from the same initial condition:

$$
\hat{\mathbf{x}}_0 = \mathbf{x}_0
$$

At short lead times, we hope the AI trajectory stays close to the physics trajectory.

At longer lead times, we expect the two trajectories to diverge because the Lorenz system is chaotic.

## 08.1 Define the AI rollout function

In [ ]:
# ============================================================
# 08.1 AI-based rollout function
# ============================================================
#
# This function uses the trained neural network as a time integrator.
#
# Starting from an initial state, it repeatedly:
#
# 1. normalizes the current state,
# 2. sends it through the neural network,
# 3. denormalizes the predicted next state,
# 4. feeds that prediction back as the next input.
#
# This is different from one-step validation, where the input always
# comes from the true physics-generated data.
# ============================================================

def rollout_ai_model(model, initial_state, num_steps):
    """
    Roll out the trained neural network emulator.

    Parameters
    ----------
    model : torch.nn.Module
        Trained Lorenz emulator.

    initial_state : numpy.ndarray of shape (3,)
        Initial Lorenz state [x0, y0, z0].

    num_steps : int
        Number of time steps to roll out.

    Returns
    -------
    trajectory_ai : numpy.ndarray of shape (num_steps + 1, 3)
        AI-generated trajectory.
    """

    # Make sure the model is in evaluation mode
    model.eval()

    # Array for storing the AI trajectory
    trajectory_ai = np.zeros((num_steps + 1, 3), dtype=np.float32)

    # Store initial condition
    trajectory_ai[0] = initial_state

    # Current state starts from the initial condition
    current_state = initial_state.copy()

    # Disable gradient computation because we are not training here
    with torch.no_grad():

        for n in range(num_steps):

            # Normalize current state
            current_state_norm = normalize_state(current_state)

            # Convert to PyTorch tensor and add batch dimension
            current_state_tensor = torch.tensor(
                current_state_norm,
                dtype=torch.float32
            ).unsqueeze(0).to(device)

            # Predict next normalized state
            next_state_norm_tensor = model(current_state_tensor)

            # Convert prediction back to NumPy
            next_state_norm = next_state_norm_tensor.cpu().numpy()[0]

            # Denormalize back to physical Lorenz units
            next_state = denormalize_state(next_state_norm)

            # Store predicted state
            trajectory_ai[n + 1] = next_state

            # Feed prediction back into the model
            current_state = next_state

    return trajectory_ai

## 08.2 Generate a physics rollout and an AI rollout

Now we choose an initial condition and compare:

1. the RK4 physics trajectory,
2. the AI-emulated trajectory.

To make the comparison fair, both trajectories start from exactly the same initial state.

We will use a short-to-moderate rollout time first. If the rollout is too long, the chaotic trajectories will eventually separate strongly.

In [ ]:
# ============================================================
# Generate physics and AI rollouts from the same initial state
# ============================================================

# Test initial condition
test_initial_state = np.array([1.0, 1.0, 1.0], dtype=np.float32)

# Rollout settings
dt = 0.01
rollout_time = 20.0
num_rollout_steps = int(rollout_time / dt)

# Physics rollout using RK4
times_rollout, trajectory_physics = simulate_lorenz_rk4(
    initial_state=test_initial_state,
    dt=dt,
    total_time=rollout_time
)

# AI rollout using the trained neural network
trajectory_ai = rollout_ai_model(
    model=model,
    initial_state=test_initial_state,
    num_steps=num_rollout_steps
)

print("Physics trajectory shape:", trajectory_physics.shape)
print("AI trajectory shape:", trajectory_ai.shape)

## 08.3 Compare time series

In [ ]:
# ============================================================
# Compare physics and AI trajectories as time series
# ============================================================
#
# We first compare x(t), y(t), and z(t) separately.
#
# This helps us see:
#
# - whether the AI model follows the physics trajectory,
# - when the trajectories begin to diverge,
# - which variable diverges most strongly.
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

variables = ["x", "y", "z"]
colors = ["r", "g", "b"]

for i, ax in enumerate(axes):
    ax.plot(
        times_rollout,
        trajectory_physics[:, i],
        color=colors[i],
        linewidth=2,
        label=f"Physics {variables[i]}(t)"
    )

    ax.plot(
        times_rollout,
        trajectory_ai[:, i],
        color=colors[i],
        linestyle="dashed",
        linewidth=2,
        label=f"AI {variables[i]}(t)"
    )

    ax.set_ylabel(f"{variables[i]} value")
    ax.legend()
    ax.grid(True)

axes[-1].set_xlabel("Time")

fig.suptitle("Physics rollout versus AI rollout", fontsize=14)
plt.tight_layout()
plt.show()

The AI rollout may follow the physics trajectory for a while.

However, after some time, the two trajectories will usually separate.

This does not necessarily mean the neural network is useless.

For chaotic systems, even very small numerical errors or model errors can grow rapidly. Therefore, long-term pointwise agreement is very difficult.

This is also true in weather prediction. Forecasts can be useful for some time, but the exact trajectory becomes increasingly uncertain at longer lead times.

## 08.4 Plot the 3D trajectories

In [ ]:
# ============================================================
# Compare physics and AI trajectories in 3D phase space
# ============================================================
#
# The Lorenz system is often visualized in phase space.
#
# Here we compare:
#
# - the physics trajectory,
# - the AI trajectory.
#
# If the AI model is good, it should remain near the Lorenz attractor,
# even if it eventually diverges from the exact physics trajectory.
# ============================================================

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    trajectory_physics[:, 0],
    trajectory_physics[:, 1],
    trajectory_physics[:, 2],
    label="Physics RK4",
    linewidth=1.5
)

ax.plot(
    trajectory_ai[:, 0],
    trajectory_ai[:, 1],
    trajectory_ai[:, 2],
    "--",
    label="AI emulator",
    linewidth=1.5
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Physics trajectory versus AI trajectory in phase space")
ax.legend()

plt.show()

## 08.5 Quantify rollout error

## Rollout error

Visual comparison is useful, but we also want a quantitative measure of error.

At each time step, we compute the Euclidean distance between the physics state and the AI state:

$$
e_n =
\left\|
\mathbf{x}_n - \hat{\mathbf{x}}_n
\right\|_2
$$

In expanded form:

$$
e_n =
\sqrt{
(x_n - \hat{x}_n)^2
+
(y_n - \hat{y}_n)^2
+
(z_n - \hat{z}_n)^2
}
$$

This gives one error value at each time step.

In [ ]:
# ============================================================
# Compute and plot rollout error
# ============================================================

# Difference between physics and AI states
trajectory_difference = trajectory_physics - trajectory_ai

# Euclidean error at each time step
rollout_error = np.linalg.norm(trajectory_difference, axis=1)

plt.figure(figsize=(10, 5))

plt.plot(times_rollout, rollout_error, linewidth=2)

plt.xlabel("Time")
plt.ylabel("Euclidean error")
plt.title("Growth of AI rollout error")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.semilogy(times_rollout, rollout_error + 1e-12, linewidth=2)

plt.xlabel("Time")
plt.ylabel("Euclidean error, log scale")
plt.title("AI rollout error growth on a logarithmic scale")
plt.grid(True)
plt.show()

The error usually starts near zero because both trajectories begin from the same initial condition.

As time increases, the error often grows.

This error growth has two sources:

1. **emulator error**  
   The neural network does not predict the exact next state perfectly.

2. **chaotic amplification**  
   The Lorenz system amplifies small differences in initial conditions and intermediate states.

This is one of the key lessons of the workshop:

> A model can have very small one-step error and still drift away during long autoregressive rollout.

## 08.6 Short rollout versus long rollout

For chaotic systems, it is useful to distinguish between:

### Short lead times

The AI model may closely follow the physics trajectory.

### Long lead times

The AI model may no longer match the exact physics trajectory point by point.

However, the AI model may still reproduce the general shape of the Lorenz attractor.

This distinction is important in weather and climate applications.

For weather prediction, the exact trajectory matters.

For climate-like statistics, the long-term distribution may also be important.

In [ ]:
# ============================================================
# Compare short and long AI rollouts
# ============================================================

# Let us try a longer rollout
long_rollout_time = 30.0
long_num_steps = int(long_rollout_time / dt)

# Physics rollout
times_long, trajectory_physics_long = simulate_lorenz_rk4(
    initial_state=test_initial_state,
    dt=dt,
    total_time=long_rollout_time
)

# AI rollout
trajectory_ai_long = rollout_ai_model(
    model=model,
    initial_state=test_initial_state,
    num_steps=long_num_steps
)

# Plot in phase space
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    trajectory_physics_long[:, 0],
    trajectory_physics_long[:, 1],
    trajectory_physics_long[:, 2],
    label="Physics RK4",
    linewidth=1.0,
    alpha=0.8
)

ax.plot(
    trajectory_ai_long[:, 0],
    trajectory_ai_long[:, 1],
    trajectory_ai_long[:, 2],
    "--",
    label="AI emulator",
    linewidth=1.0,
    alpha=0.8
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Long rollout: physics versus AI")
ax.legend()

plt.show()

For the long rollout, the AI trajectory may no longer follow the exact same path as the physics trajectory.

This is expected.

The more important question becomes:

> Does the AI trajectory remain on a physically reasonable Lorenz-like attractor?

If the AI trajectory explodes to very large values, collapses to a fixed point, or leaves the attractor completely, then the emulator is not stable as a long-term time-integration model.

If it stays on a similar attractor, then it has learned something useful about the system dynamics, even if it cannot reproduce the exact chaotic trajectory forever.

## 08.7 Compare distributions instead of exact trajectories

For chaotic systems, exact point-by-point trajectory matching becomes difficult at long lead times.

Therefore, we can also compare statistical properties.

For example, we can compare the distribution of $x$, $y$, and $z$ values from:

- the physics trajectory,
- the AI trajectory.

This is similar to the difference between weather and climate:

- weather prediction asks for the exact future trajectory,
- climate analysis often asks whether the long-term statistics are realistic.

In [ ]:
# ============================================================
# Compare distributions of physics and AI trajectories
# ============================================================
#
# Here we compare histograms of x, y, and z values.
#
# This tells us whether the AI model visits similar regions
# of state space as the physics model.
# ============================================================

variables = ["x", "y", "z"]

for i, var_name in enumerate(variables):

    plt.figure(figsize=(8, 5))

    plt.hist(
        trajectory_physics_long[:, i],
        bins=50,
        alpha=0.6,
        density=True,
        label=f"Physics {var_name}"
    )

    plt.hist(
        trajectory_ai_long[:, i],
        bins=50,
        alpha=0.6,
        density=True,
        label=f"AI {var_name}"
    )

    plt.xlabel(var_name)
    plt.ylabel("Density")
    plt.title(f"Distribution comparison for {var_name}")
    plt.legend()
    plt.grid(True)
    plt.show()

The histograms compare the values visited by the physics model and the AI model.

Even if the exact trajectories diverge, the distributions may still look similar.

This distinction is important:

- **trajectory accuracy** asks whether the model follows the exact path,
- **statistical accuracy** asks whether the model produces realistic long-term behavior.

For chaotic systems, both perspectives are useful.

# 09. Sensitivity to initial conditions: physics versus AI

Earlier, we studied sensitivity to initial conditions using the physics-based Lorenz model.

We started from two almost identical initial conditions and showed that the trajectories eventually separate.

Now we ask the same question for the trained AI emulator.

If the AI emulator has learned the Lorenz dynamics well, then it should not only predict one-step transitions accurately. It should also reproduce the qualitative error-growth behavior of the Lorenz system.

In other words, we ask:

> Does the AI model amplify small initial perturbations in a similar way to the physics model?

This is important because, in meteorology, forecast uncertainty is strongly connected to the growth of small initial-condition errors.

## Two nearby initial conditions

We use two initial states:

$$
\mathbf{x}_0 =
\begin{bmatrix}
1.0 \\
1.0 \\
1.0
\end{bmatrix}
$$

and

$$
\mathbf{x}_0^\prime =
\begin{bmatrix}
1.0 + 10^{-6} \\
1.0 \\
1.0
\end{bmatrix}
$$

The difference between them is tiny.

We will compare how this small perturbation grows under:

1. RK4 physics integration,
2. AI emulator rollout.

In [ ]:
# ============================================================
# 09. Sensitivity to initial conditions: physics versus AI
# ============================================================
#
# We compare how a tiny perturbation in the initial condition
# grows under:
#
# 1. The physics model using RK4,
# 2. The trained AI emulator.
#
# This helps us test whether the AI emulator reproduces the
# qualitative sensitivity of the Lorenz system.
# ============================================================

# Two nearly identical initial conditions
initial_state_a = np.array([1.0, 1.0, 1.0], dtype=np.float32)
initial_state_b = np.array([1.0 + 1e-6, 1.0, 1.0], dtype=np.float32)

# Rollout settings
dt = 0.01
sensitivity_time = 20.0
num_sensitivity_steps = int(sensitivity_time / dt)

# Physics rollouts
times_sensitivity, physics_a = simulate_lorenz_rk4(
    initial_state=initial_state_a,
    dt=dt,
    total_time=sensitivity_time
)

_, physics_b = simulate_lorenz_rk4(
    initial_state=initial_state_b,
    dt=dt,
    total_time=sensitivity_time
)

# AI rollouts
ai_a = rollout_ai_model(
    model=model,
    initial_state=initial_state_a,
    num_steps=num_sensitivity_steps
)

ai_b = rollout_ai_model(
    model=model,
    initial_state=initial_state_b,
    num_steps=num_sensitivity_steps
)

## 09.1 Plot the perturbed trajectories

In [ ]:
# ============================================================
# Plot x(t) for nearby initial conditions: physics and AI
# ============================================================

plt.figure(figsize=(12, 6))

# Physics-based trajectories
plt.plot(
    times_sensitivity,
    physics_a[:, 0],
    color="tab:blue",
    linestyle="-",
    label="Physics: initial state A",
    linewidth=2
)

plt.plot(
    times_sensitivity,
    physics_b[:, 0],
    color="tab:blue",
    linestyle="--",
    label="Physics: initial state B",
    linewidth=2
)

# AI-based trajectories
plt.plot(
    times_sensitivity,
    ai_a[:, 0],
    color="tab:orange",
    linestyle="-",
    label="AI: initial state A",
    linewidth=2,
    alpha=0.85
)

plt.plot(
    times_sensitivity,
    ai_b[:, 0],
    color="tab:orange",
    linestyle="--",
    label="AI: initial state B",
    linewidth=2,
    alpha=0.85
)

plt.xlabel("Time")
plt.ylabel("x(t)")
plt.title("Sensitivity to initial conditions: physics versus AI")
plt.legend()
plt.grid(True)
plt.show()

## 09.2 Perturbation growth

For both the physics model and the AI emulator, we compute the distance between the two perturbed trajectories:

$$
d(t) =
\left\|
\mathbf{x}_A(t) - \mathbf{x}_B(t)
\right\|_2
$$

For the physics model:

$$
d_{physics}(t)
=
\left\|
\mathbf{x}_{physics,A}(t)
-
\mathbf{x}_{physics,B}(t)
\right\|_2
$$

For the AI model:

$$
d_{AI}(t)
=
\left\|
\mathbf{x}_{AI,A}(t)
-
\mathbf{x}_{AI,B}(t)
\right\|_2
$$

If the AI emulator reproduces the chaotic sensitivity of the Lorenz system well, the early-time growth of $d_{AI}(t)$ should be qualitatively similar to $d_{physics}(t)$.

In [ ]:
# ============================================================
# Compute perturbation growth for physics and AI
# ============================================================

# Distance between the two physics trajectories
distance_physics = np.linalg.norm(physics_a - physics_b, axis=1)

# Distance between the two AI trajectories
distance_ai = np.linalg.norm(ai_a - ai_b, axis=1)

plt.figure(figsize=(10, 5))

plt.plot(
    times_sensitivity,
    distance_physics,
    label="Physics perturbation growth",
    linewidth=2
)

plt.plot(
    times_sensitivity,
    distance_ai,
    "--",
    label="AI perturbation growth",
    linewidth=2
)

plt.xlabel("Time")
plt.ylabel("Distance between perturbed trajectories")
plt.title("Perturbation growth: physics versus AI")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# Log-scale perturbation growth
# ============================================================
#
# The early growth of perturbations in a chaotic system is often
# approximately exponential.
#
# A logarithmic y-axis makes exponential-like growth appear
# approximately linear.
# ============================================================

plt.figure(figsize=(10, 5))

plt.semilogy(
    times_sensitivity,
    distance_physics + 1e-12,
    label="Physics perturbation growth",
    linewidth=2
)

plt.semilogy(
    times_sensitivity,
    distance_ai + 1e-12,
    "--",
    label="AI perturbation growth",
    linewidth=2
)

plt.xlabel("Time")
plt.ylabel("Distance between perturbed trajectories, log scale")
plt.title("Perturbation growth on a log scale")
plt.legend()
plt.grid(True)
plt.show()

The log-scale plot is especially useful.

At early times, both curves may show approximately exponential growth. If the AI emulator is dynamically faithful, its perturbation-growth curve should roughly follow the physics curve at short lead times.

At longer lead times, both distances may saturate because the two trajectories are no longer close. Once the trajectories are on very different parts of the attractor, the distance cannot keep growing exponentially forever.

If the AI curve grows much faster than the physics curve, the emulator may be too unstable.

If the AI curve grows much more slowly, the emulator may be overly damped or may have learned smoother dynamics than the true Lorenz system.

If the AI curve collapses toward zero, the emulator may be artificially attracting trajectories together, which would be dynamically unrealistic.

## 09.3 Short-term versus long-term behavior

For chaotic systems, it is useful to separate two questions.

### Short-term sensitivity

Does the AI emulator amplify small initial differences at approximately the same rate as the physics model?

This is related to whether the AI model has learned the local instability of the Lorenz dynamics.

### Long-term behavior

After the trajectories have separated, do they remain on a realistic Lorenz-like attractor?

At long lead times, exact trajectory agreement is not expected. Instead, we ask whether the model preserves the qualitative geometry and statistics of the system.

In [ ]:
# ============================================================
# Short-term zoom of perturbation growth
# ============================================================
#
# We focus on the early period before the trajectories are fully
# separated. This is where the perturbation growth rate is most
# meaningful.
# ============================================================

short_time_limit = 8.0

short_mask = times_sensitivity <= short_time_limit

plt.figure(figsize=(10, 5))

plt.semilogy(
    times_sensitivity[short_mask],
    distance_physics[short_mask] + 1e-12,
    label="Physics perturbation growth",
    linewidth=2
)

plt.semilogy(
    times_sensitivity[short_mask],
    distance_ai[short_mask] + 1e-12,
    "--",
    label="AI perturbation growth",
    linewidth=2
)

plt.xlabel("Time")
plt.ylabel("Distance, log scale")
plt.title("Short-term perturbation growth")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# Long-term behavior of perturbed AI and physics trajectories
# ============================================================

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot(
    physics_a[:, 0],
    physics_a[:, 1],
    physics_a[:, 2],
    label="Physics A",
    linewidth=1.0,
    alpha=0.8
)

ax.plot(
    physics_b[:, 0],
    physics_b[:, 1],
    physics_b[:, 2],
    "--",
    label="Physics B",
    linewidth=1.0,
    alpha=0.8
)

ax.plot(
    ai_a[:, 0],
    ai_a[:, 1],
    ai_a[:, 2],
    label="AI A",
    linewidth=1.0,
    alpha=0.8
)

ax.plot(
    ai_b[:, 0],
    ai_b[:, 1],
    ai_b[:, 2],
    "--",
    label="AI B",
    linewidth=1.0,
    alpha=0.8
)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("Long-term behavior of perturbed trajectories")
ax.legend()

plt.show()

## 09.4 Optional: estimating an early error-growth rate

In a chaotic system, small perturbations often grow approximately exponentially for some initial period:

$$
d(t) \approx d(0) e^{\lambda t}
$$

Taking the logarithm gives:

$$
\log d(t) \approx \log d(0) + \lambda t
$$

So, during the early growth phase, the slope of $\log d(t)$ gives an approximate error-growth rate.

This is related to the idea of a Lyapunov exponent.

Here we estimate this growth rate over a short time window.

In [ ]:
# ============================================================
# Optional: estimate early perturbation growth rate
# ============================================================
#
# This is a simple approximate diagnostic.
#
# We choose a short time window where the perturbation is still small
# and fit a straight line to log(distance).
# ============================================================

def estimate_growth_rate(times, distances, t_min=0.0, t_max=5.0):
    """
    Estimate an approximate early error-growth rate.

    Parameters
    ----------
    times : numpy.ndarray
        Time array.

    distances : numpy.ndarray
        Distance between perturbed trajectories.

    t_min, t_max : float
        Time window used for the fit.

    Returns
    -------
    growth_rate : float
        Estimated slope of log(distance) versus time.

    intercept : float
        Intercept of the fitted line.
    """

    # Select time window
    mask = (times >= t_min) & (times <= t_max)

    t_fit = times[mask]
    d_fit = distances[mask]

    # Avoid taking log of zero
    d_fit = np.maximum(d_fit, 1e-12)

    # Fit log(distance) = intercept + growth_rate * time
    growth_rate, intercept = np.polyfit(t_fit, np.log(d_fit), 1)

    return growth_rate, intercept


physics_growth_rate, physics_intercept = estimate_growth_rate(
    times_sensitivity,
    distance_physics,
    t_min=0.0,
    t_max=5.0
)

ai_growth_rate, ai_intercept = estimate_growth_rate(
    times_sensitivity,
    distance_ai,
    t_min=0.0,
    t_max=5.0
)

print("Approximate early perturbation growth rate")
print("------------------------------------------")
print(f"Physics RK4: {physics_growth_rate:.3f}")
print(f"AI emulator: {ai_growth_rate:.3f}")

The estimated growth rates should be interpreted carefully.

They depend on the chosen time window and on the specific initial condition. This is not a rigorous Lyapunov exponent calculation.

However, it is a useful workshop diagnostic.

If the AI emulator has learned the local instability of the Lorenz system well, its early perturbation growth rate should be reasonably close to the physics-based estimate.

## For a better emulator - 

The Lorenz attractor has two main lobes. For training an AI emulator, it is useful to include samples from both lobes in the dataset.

In the dataset generation function, we sample many random initial conditions and integrate them forward. This usually produces samples from both lobes, but it does not strictly guarantee perfect balance.

We also use a burn-in period. During burn-in, the system is integrated forward, but the states are not stored. This removes the early transient part of the trajectory and helps us collect samples closer to the long-term attractor.

A simple diagnostic is to count how many samples have positive and negative values of $x$ or $y$.

For learning the long-term Lorenz dynamics, we often want to remove this initial transient.

1. sample random initial condition
2. Run the system for some burn-in steps
3. Discard those burn-in states
4. Then collect training pairs

In [ ]:
def generate_lorenz_dataset(num_trajectories=200,
                            steps_per_trajectory=300,
                            burn_in_steps=500,
                            dt=0.01,
                            initial_range=(-20.0, 20.0),
                            sigma=10.0,
                            rho=28.0,
                            beta=8.0/3.0,
                            seed=42):
    """
    Generate supervised learning data from the Lorenz system.

    Each training example is:

        input  = current state  [x_n, y_n, z_n]
        target = next state     [x_{n+1}, y_{n+1}, z_{n+1}]

    We first run each trajectory for a number of burn-in steps.
    These burn-in states are discarded.

    Why burn-in?

    Random initial conditions may start far away from the Lorenz attractor.
    After some time, the trajectory approaches the attractor. Since we mainly
    want the neural network to learn the typical long-term Lorenz dynamics,
    we discard the early transient part.

    Parameters
    ----------
    num_trajectories : int
        Number of independent trajectories to simulate.

    steps_per_trajectory : int
        Number of training pairs collected from each trajectory.

    burn_in_steps : int
        Number of initial integration steps to discard before collecting data.

    dt : float
        Time step size.

    initial_range : tuple of float
        Range from which initial x, y, z values are sampled uniformly.

    sigma, rho, beta : float
        Lorenz parameters.

    seed : int
        Random seed for reproducibility.

    Returns
    -------
    X : numpy.ndarray of shape (num_samples, 3)
        Input states.

    Y : numpy.ndarray of shape (num_samples, 3)
        Target next states.
    """

    rng = np.random.default_rng(seed)

    inputs = []
    targets = []

    for i in range(num_trajectories):

        # Random initial condition
        state = rng.uniform(
            low=initial_range[0],
            high=initial_range[1],
            size=3
        )

        # ----------------------------------------------------
        # Burn-in phase
        # ----------------------------------------------------
        # We integrate forward without storing the states.
        # This allows the trajectory to approach the attractor.
        # ----------------------------------------------------
        for _ in range(burn_in_steps):
            state = rk4_step(
                state,
                dt,
                sigma=sigma,
                rho=rho,
                beta=beta
            )

        # ----------------------------------------------------
        # Data collection phase
        # ----------------------------------------------------
        # Now we store input-target pairs from the trajectory.
        # ----------------------------------------------------
        for _ in range(steps_per_trajectory):

            next_state = rk4_step(
                state,
                dt,
                sigma=sigma,
                rho=rho,
                beta=beta
            )

            inputs.append(state)
            targets.append(next_state)

            state = next_state

    X = np.array(inputs, dtype=np.float32)
    Y = np.array(targets, dtype=np.float32)

    return X, Y

In [ ]:
X, Y = generate_lorenz_dataset(
    num_trajectories=200,
    steps_per_trajectory=300,
    burn_in_steps=500,
    dt=0.01,
    initial_range=(-20.0, 20.0),
    seed=42
)

print("Input data shape:", X.shape)
print("Target data shape:", Y.shape)

#### simple_diagnostics to check how many samples have positive and negative x and y

In [ ]:
num_positive_x = np.sum(X[:, 0] > 0)
num_negative_x = np.sum(X[:, 0] < 0)

print("Samples with x > 0:", num_positive_x)
print("Samples with x < 0:", num_negative_x)

print("Fraction with x > 0:", num_positive_x / len(X))
print("Fraction with x < 0:", num_negative_x / len(X))

num_positive_y = np.sum(X[:, 1] > 0)
num_negative_y = np.sum(X[:, 1] < 0)

print("Samples with y > 0:", num_positive_y)
print("Samples with y < 0:", num_negative_y)

print("Fraction with y > 0:", num_positive_y / len(X))
print("Fraction with y < 0:", num_negative_y / len(X))